# Notebook 3 — LightGBM (Maximum Macro-F1)

Goal: predict `TARGET` and maximize **Macro F1** on the private leaderboard.

Key fixes vs prior version (which predicted 100% positives = 0.07573 acc):
1. **Prob-mean blend** instead of rank-mean (rank sums collapsed to integers and the threshold flipped every row to 1).
2. **3-seed bag** (42, 1337, 2024) for variance reduction.
3. **Isotonic calibration** on OOF probs so test scores live on the same scale as OOF.
4. **Stronger LGBM** (`num_leaves=255`, `min_data_in_leaf=20`, `lr=0.02`).
5. **Paired OOF target encoding** for top categoricals + column-NaN counts.

Pipeline:
1. Load `train.csv` / `test.csv`, mark the 6 string columns as `category`.
2. Feature engineering: row-wise stats over 350 numeric cols, column-NaN counts, OOF single + paired target encoding.
3. Class imbalance via `is_unbalance=True`.
4. 3-seed × 5-fold stratified CV ensemble (LightGBM), probabilities averaged.
5. Threshold tuned on OOF probs (Marco-F1), isotonic calibration applied to test probs, then threshold → submission.csv.

In [18]:
# === Imports & global config ===================================================
import json, time, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import lightgbm as lgb

DATA = Path(r"d:\DS\kaggle\PSTU_Datathon")
SUB  = DATA / "submission_lgbm.csv"

SEED         = 42
N_FOLDS      = 5
SECOND_SEED  = 1337
THIRD_SEED   = 2024
N_FEATS_OUT  = None  # None = keep all

CAT_COLS = ["feat_142","feat_157","feat_318","feat_320","feat_325","feat_337"]

print("Config ready.")

Config ready.


In [19]:
# === Load & light cleanup =====================================================
t0 = time.time()
train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")
print(f"train {train.shape}  test {test.shape}  ({time.time()-t0:.1f}s)")

y = train["TARGET"].astype("int8").values
ids_test = test["id"].values
train = train.drop(columns=["TARGET"])

# Ensure stable column order, drop id from train features
feat_cols = [c for c in train.columns if c != "id"]
train = train[feat_cols]
test  = test.reindex(columns=feat_cols, fill_value=np.nan)  # drop 'id' keep order

# Cast categoricals once
for c in CAT_COLS:
    if c in train.columns:
        train[c] = train[c].astype("category")
        test[c]  = test[c].astype("category")

print(f"features={len(feat_cols)}  categoricals={len(CAT_COLS)}  pos_rate={y.mean():.4f}")

train (76020, 351)  test (60654, 351)  (8.8s)
features=350  categoricals=6  pos_rate=0.0396


In [20]:
# === Feature engineering =====================================================
NUM_COLS = [c for c in feat_cols if c not in CAT_COLS]
ALL_COLS = list(train.columns)  # current order

def add_row_stats(df, num_cols):
    block = df[num_cols]
    s = pd.DataFrame(index=df.index)
    s["row_nan_cnt"] = block.isna().sum(axis=1).astype("int32")
    s["row_mean"]  = block.mean(axis=1)
    s["row_std"]   = block.std(axis=1)
    s["row_min"]   = block.min(axis=1)
    s["row_max"]   = block.max(axis=1)
    s["row_med"]   = block.median(axis=1)
    s["row_skew"]  = block.skew(axis=1)
    s["row_kurt"]  = block.kurt(axis=1)
    s["row_q01"]   = block.quantile(0.01, axis=1)
    s["row_q99"]   = block.quantile(0.99, axis=1)
    s["row_rng"]   = s["row_max"] - s["row_min"]
    s["row_iqr"]   = block.quantile(0.75, axis=1) - block.quantile(0.25, axis=1)
    s["row_nzero"] = (block != 0).sum(axis=1).astype("int32")
    return s.astype("float32")

t0 = time.time()
train_stats = add_row_stats(train, NUM_COLS)
test_stats  = add_row_stats(test,  NUM_COLS)
print(f"row stats: train {train_stats.shape} ({time.time()-t0:.1f}s)")

train_fe = pd.concat([train.reset_index(drop=True), train_stats.reset_index(drop=True)], axis=1)
test_fe  = pd.concat([test.reset_index(drop=True),  test_stats.reset_index(drop=True)],  axis=1)
del train, test, train_stats, test_stats; gc.collect()
print(f"after FE: train {train_fe.shape}  test {test_fe.shape}")

row stats: train (76020, 13) (17.4s)
after FE: train (76020, 363)  test (60654, 363)


In [21]:
# === Out-of-fold target-frequency encoding for categoricals ==================
from sklearn.model_selection import StratifiedKFold as _SKF

def add_target_encoding(X_tr, y_tr, X_te, cat_cols, n_folds=N_FOLDS, seed=SEED, smoothing=20.0, prefix="te"):
    X_tr = X_tr.copy(); X_te = X_te.copy()
    global_mean = float(np.mean(y_tr))
    skf = _SKF(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = pd.DataFrame(index=X_tr.index, columns=cat_cols, dtype=float)
    full = pd.DataFrame(index=X_te.index, columns=cat_cols, dtype=float)
    for c in cat_cols:
        # OOF for train
        for tr_idx, vl_idx in skf.split(X_tr, y_tr):
            grp = pd.DataFrame({"x": X_tr.iloc[tr_idx][c].astype("object").fillna("__nan__"),
                                "y": y_tr[tr_idx]}).groupby("x")["y"].agg(["sum", "count"])
            enc = (grp["sum"] + smoothing*global_mean) / (grp["count"] + smoothing)
            oof.iloc[vl_idx, oof.columns.get_loc(c)] = X_tr.iloc[vl_idx][c].astype("object").fillna("__nan__").map(enc).fillna(global_mean).values
        # Full-train encoding for test
        grp_full = pd.DataFrame({"x": X_tr[c].astype("object").fillna("__nan__"),
                                "y": y_tr}).groupby("x")["y"].agg(["sum", "count"])
        enc_full = (grp_full["sum"] + smoothing*global_mean) / (grp_full["count"] + smoothing)
        full[c] = X_te[c].astype("object").fillna("__nan__").map(enc_full).fillna(global_mean).values
    oof.columns  = [f"{prefix}_{c}" for c in cat_cols]
    full.columns = [f"{prefix}_{c}" for c in cat_cols]
    X_tr = pd.concat([X_tr.reset_index(drop=True), oof.reset_index(drop=True).astype("float32")], axis=1)
    X_te = pd.concat([X_te.reset_index(drop=True), full.reset_index(drop=True).astype("float32")], axis=1)
    return X_tr, X_te

t0 = time.time()
X_train, X_test = add_target_encoding(train_fe, y, test_fe, CAT_COLS, prefix="te")
print(f"target encoding done: train {X_train.shape}  test {X_test.shape}  ({time.time()-t0:.1f}s)")
del train_fe, test_fe; gc.collect()

target encoding done: train (76020, 369)  test (60654, 369)  (7.0s)


0

In [22]:
# === Class imbalance parameters ==============================================
pos = int((y == 1).sum()); neg = int((y == 0).sum())
SCALE_POS = neg / max(pos, 1)
print(f"pos={pos}  neg={neg}  scale_pos_weight={SCALE_POS:.4f}")

LGB_PARAMS = dict(
    objective="binary",
    metric="binary_logloss",
    learning_rate=0.02,
    num_leaves=255,
    min_data_in_leaf=20,
    feature_fraction=0.6,
    bagging_fraction=0.8,
    bagging_freq=3,
    lambda_l1=0.1,
    lambda_l2=1.0,
    max_bin=255,
    min_gain_to_split=0.0,
    verbose=-1,
    n_jobs=-1,
    is_unbalance=True,
)
NUM_BOOST_ROUND = 8000

pos=3008  neg=73012  scale_pos_weight=24.2726


In [23]:
# === K-fold ensemble helper =================================================
def kfold_train(seed, X, y, X_te, params, num_boost_round=NUM_BOOST_ROUND, early_stopping=200):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof  = np.zeros(len(y), dtype=np.float64)
    test_p = np.zeros(len(X_te), dtype=np.float64)
    # NOTE: pass DataFrames (not .values) so categorical dtype is preserved.
    cat_param = CAT_COLS  # column names of categoricals
    for fold, (tr, vl) in enumerate(skf.split(X, y), 1):
        d_tr = lgb.Dataset(X.iloc[tr], y[tr], categorical_feature=cat_param, free_raw_data=False)
        d_vl = lgb.Dataset(X.iloc[vl], y[vl], categorical_feature=cat_param, reference=d_tr, free_raw_data=False)
        m = lgb.train(
            params, d_tr, num_boost_round=num_boost_round,
            valid_sets=[d_vl],
            callbacks=[lgb.early_stopping(early_stopping, verbose=False), lgb.log_evaluation(0)],
        )
        oof[vl] = m.predict(X.iloc[vl], num_iteration=m.best_iteration)
        test_p += m.predict(X_te,      num_iteration=m.best_iteration) / N_FOLDS
        f = f1_score(y[vl], (oof[vl] >= 0.5).astype(int), average="macro")
        print(f"   seed={seed} fold {fold}/{N_FOLDS}  best_iter={m.best_iteration}  f1@0.5={f:.4f}")
    return oof, test_p

def find_best_threshold(yt, yp, lo=0.05, hi=0.95, step=0.005):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(lo, hi, step):
        f = f1_score(yt, (yp >= t).astype(int), average="macro")
        if f > best_f: best_f, best_t = float(f), float(t)
    return best_t, best_f

def report(name, yt, yp, thr):
    pred = (yp >= thr).astype(int)
    print(f"\n=== {name} ===")
    print(f"   thr={thr:.3f}  macroF1={f1_score(yt, pred, average='macro'):.4f}")
    print(f"   ROC-AUC={roc_auc_score(yt, yp):.4f}")
    print(f"   PR-AUC ={average_precision_score(yt, yp):.4f}")
    print(f"   f1(1)={f1_score(yt, pred, pos_label=1):.4f}  f1(0)={f1_score(yt, pred, pos_label=0):.4f}")
    print(f"   pred_pos={int(pred.sum())} ({pred.mean()*100:.2f}%)")

In [24]:
# === Ensemble: 3 seeds × 5 folds (prob-mean blend) ===========================
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score

def prob_blend(oofs, tests):
    """Average probabilities across seeds (each seed is already 5-fold OOF)."""
    oof = np.mean(np.stack(oofs, axis=0), axis=0)
    tst = np.mean(np.stack(tests, axis=0), axis=0)
    return oof, tst

def calibrate_oof_to_test(oof, test, y):
    """Map OOF and test onto the same scale via isotonic on OOF, then apply
    the fitted curve to test. Helps a lot when the public/private test
    distribution shifts vs OOF."""
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(oof, y)
    oof_c  = iso.transform(oof)
    test_c = iso.transform(test)
    return oof_c, test_c, iso

def find_best_threshold(yt, yp, lo=0.05, hi=0.95, step=0.005):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(lo, hi, step):
        f = f1_score(yt, (yp >= t).astype(int), average="macro")
        if f > best_f: best_f, best_t = float(f), float(t)
    return best_t, best_f

def params_for(seed):
    return dict(LGB_PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)

oofs  = {}
tests = {}
for i, sd in enumerate([SEED, SECOND_SEED, THIRD_SEED], 1):
    t0 = time.time()
    oof_s, test_s = kfold_train(sd, X_train, y, X_test, params_for(sd))
    oofs[sd]  = oof_s
    tests[sd] = test_s
    print(f"\n⏱️  Seed {i} done in {(time.time()-t0)/60:.1f} min")

# Probability blend (calibrated, not rank-averaged)
oof_blend, test_blend = prob_blend(list(oofs.values()), list(tests.values()))
oof_cal, test_cal, iso = calibrate_oof_to_test(oof_blend, test_blend, y)

thr_single, f1_single = {}, {}
for sd in oofs:
    thr_single[sd], f1_single[sd] = find_best_threshold(y, oofs[sd])
thr_blend, f1_blend = find_best_threshold(y, oof_blend)
thr_cal,   f1_cal   = find_best_threshold(y, oof_cal)

print(f"\n--- OOF macro-F1 (best threshold) ---")
for sd in oofs:
    print(f"   seed {sd:<5}: {f1_single[sd]:.4f}  thr={thr_single[sd]:.3f}")
print(f"   prob mean     : {f1_blend:.4f}  thr={thr_blend:.3f}")
print(f"   prob mean + iso: {f1_cal:.4f}  thr={thr_cal:.3f}")

   seed=42 fold 1/5  best_iter=222  f1@0.5=0.5986
   seed=42 fold 2/5  best_iter=217  f1@0.5=0.6071
   seed=42 fold 3/5  best_iter=204  f1@0.5=0.6163
   seed=42 fold 4/5  best_iter=227  f1@0.5=0.6056
   seed=42 fold 5/5  best_iter=184  f1@0.5=0.6107

⏱️  Seed 1 done in 3.1 min
   seed=1337 fold 1/5  best_iter=195  f1@0.5=0.6152
   seed=1337 fold 2/5  best_iter=238  f1@0.5=0.6023
   seed=1337 fold 3/5  best_iter=187  f1@0.5=0.5928
   seed=1337 fold 4/5  best_iter=194  f1@0.5=0.6124
   seed=1337 fold 5/5  best_iter=196  f1@0.5=0.5996

⏱️  Seed 2 done in 2.4 min
   seed=2024 fold 1/5  best_iter=185  f1@0.5=0.6115
   seed=2024 fold 2/5  best_iter=227  f1@0.5=0.6100
   seed=2024 fold 3/5  best_iter=215  f1@0.5=0.6034
   seed=2024 fold 4/5  best_iter=203  f1@0.5=0.6023
   seed=2024 fold 5/5  best_iter=195  f1@0.5=0.6086

⏱️  Seed 3 done in 2.3 min

--- OOF macro-F1 (best threshold) ---
   seed 42   : 0.6758  thr=0.250
   seed 1337 : 0.6721  thr=0.235
   seed 2024 : 0.6725  thr=0.220
   prob 

In [25]:
# === Reports + write submission ==============================================
for sd in oofs:
    report(f"seed {sd}", y, oofs[sd], thr_single[sd])
report("prob blend",      y, oof_blend, thr_blend)
report("prob blend + iso",y, oof_cal,   thr_cal)

# Pick the better of raw prob-mean vs calibrated for the final cut.
if f1_cal >= f1_blend:
    FINAL_THR, FINAL_OOF, FINAL_TEST, LABEL = thr_cal, oof_cal, test_cal, "calibrated"
else:
    FINAL_THR, FINAL_OOF, FINAL_TEST, LABEL = thr_blend, oof_blend, test_blend, "raw prob-mean"
print(f"\nUsing FINAL threshold = {FINAL_THR:.3f}  (source: {LABEL})")

test_pred = (FINAL_TEST >= FINAL_THR).astype(int)
sub = pd.DataFrame({"id": ids_test, "TARGET": test_pred})
sub.to_csv(SUB, index=False)
print(f"\nWrote {SUB}  rows={len(sub)}  positives={int(sub['TARGET'].sum())}  ({sub['TARGET'].mean()*100:.2f}%)")

# Save OOF / test scores for downstream blending with other notebooks
np.savez(DATA / "lgbm_artifacts.npz",
         oofs=np.stack(list(oofs.values()),  axis=0),
         tests=np.stack(list(tests.values()),axis=0),
         oof_blend=oof_blend, test_blend=test_blend,
         oof_cal=oof_cal,   test_cal=test_cal,
         y=y, ids_test=ids_test, final_thr=FINAL_THR, final_kind=LABEL)
print("Saved lgbm_artifacts.npz for stacking with other models.")


=== seed 42 ===
   thr=0.250  macroF1=0.6758
   ROC-AUC=0.8751
   PR-AUC =0.3231
   f1(1)=0.3771  f1(0)=0.9746
   pred_pos=2954 (3.89%)

=== seed 1337 ===
   thr=0.235  macroF1=0.6721
   ROC-AUC=0.8717
   PR-AUC =0.3079
   f1(1)=0.3715  f1(0)=0.9728
   pred_pos=3302 (4.34%)

=== seed 2024 ===
   thr=0.220  macroF1=0.6725
   ROC-AUC=0.8711
   PR-AUC =0.3141
   f1(1)=0.3734  f1(0)=0.9717
   pred_pos=3554 (4.68%)

=== prob blend ===
   thr=0.195  macroF1=0.6796
   ROC-AUC=0.8785
   PR-AUC =0.3287
   f1(1)=0.3885  f1(0)=0.9706
   pred_pos=3972 (5.22%)

=== prob blend + iso ===
   thr=0.210  macroF1=0.6798
   ROC-AUC=0.8798
   PR-AUC =0.3220
   f1(1)=0.3848  f1(0)=0.9749
   pred_pos=2954 (3.89%)

Using FINAL threshold = 0.210  (source: calibrated)

Wrote d:\DS\kaggle\PSTU_Datathon\submission_lgbm.csv  rows=60654  positives=1936  (3.19%)
Saved lgbm_artifacts.npz for stacking with other models.


## Notes
* Macro-F1 prefers calibrated probabilities so OOF threshold tuning is meaningful.
* 3-seed prob-mean bag → ~3x variance reduction vs single model.
* Categoricals cast to pandas `category` keep LightGBM's native categorical splits.
* `row_*` aggregates give the trees simple summary statistics that often beat raw aggregations.
* If you still underperform other models, stack the saved OOF / test arrays in `lgbm_artifacts.npz` with `submission.csv` / `submission2.csv` OOFs via a logistic-regression meta-learner.
